In [ ]:
# Cell 1: imports
from ultralytics import YOLO
import torch, cv2, numpy as np
from PIL import Image, ImageDraw
from pytorch_grad_cam import GradCAM
import timm
# load models
yolo = YOLO("runs/detect/train/weights/best.pt")  # path contoh
cls = timm.create_model("resnet50", pretrained=False, num_classes=2)
cls.load_state_dict(torch.load("runs/cls/best.pt", map_location='cpu'))
cls.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Cell 2: helpers
def yolo_infer(image_path, conf=0.25):
    res = yolo.predict(source=image_path, conf=conf, verbose=False)[0]
    boxes = []
    for r in res.boxes.data.tolist():
        x1,y1,x2,y2,score,cls_id = r[0],r[1],r[2],r[3],r[4],int(r[5])
        boxes.append({'bbox':[x1,y1,x2-x1,y2-y1], 'score':score, 'class':cls_id})
    return boxes

def crop_and_preprocess(img_path, bbox, size=(224,224)):
    im = Image.open(img_path).convert('RGB')
    x,y,w,h = bbox
    crop = im.crop((x,y,x+w,y+h)).resize(size)
    arr = np.array(crop).astype(np.float32)/255.0
    arr = (arr - np.array([0.485,0.456,0.406]))/np.array([0.229,0.224,0.225])
    tensor = torch.tensor(arr).permute(2,0,1).unsqueeze(0)
    return tensor


In [ ]:
# Cell 3: run on sample image
img_path = "data/processed/images_640/IMG0001.jpg"
boxes = yolo_infer(img_path, conf=0.3)
print("detections:", boxes)
# draw boxes
im = Image.open(img_path).convert("RGB")
draw = ImageDraw.Draw(im)
for b in boxes:
    x,y,w,h = b['bbox']
    draw.rectangle([x,y,x+w,y+h], outline="lime", width=3)
display(im)


In [ ]:
# Cell 4: classify and gradcam
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

for b in boxes:
    tensor = crop_and_preprocess(img_path, b['bbox']).to(device)
    with torch.no_grad():
        logits = cls(tensor)
        prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    print("class prob", prob)
    # GradCAM
    target_layer = cls.get_classifier() if hasattr(cls, "get_classifier") else cls.layer4
    cam = GradCAM(model=cls, target_layers=[target_layer], use_cuda=(device=="cuda"))
    grayscale_cam = cam(input_tensor=tensor)[0]
    img_rgb = (tensor[0].permute(1,2,0).cpu().numpy() * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406]))
    cam_image = show_cam_on_image(img_rgb, grayscale_cam, use_rgb=True)
    display(Image.fromarray((cam_image*255).astype('uint8')))


In [ ]:
# Cell 5: save results
import json
out = {"image": img_path, "detections": boxes}
with open("experiments/infer_result.json","w") as f:
    json.dump(out, f, indent=2)
print("saved experiments/infer_result.json")
